## Сентимент новости по названию компании

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
from collections import Counter


# Модель от Сбера
model_name = "sberbank-ai/rugpt3small_based_on_gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Анализ одного фрагмента с фокусом на акцию
def classify_sentiment_for_stock(text, stock_name):
    prompt = f"""
Определи, в каком ключе упоминается компания "{stock_name}". Прочитай новость и ответь только одним словом — позитивный, нейтральный или негативный. 

Новость:
\"\"\"{text}\"\"\"

Ответ:
Компания {stock_name} упоминается в"""

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1800).to(device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    result = tokenizer.decode(output[0], skip_special_tokens=True).lower()

    for label in ["позитивном", "нейтральном", "негативном"]:
        if label in result:
            return label.replace("ном", "ный").capitalize()

    return "Не определено"

# Разбивка длинного текста по предложениям
def split_text(text, max_tokens=1500):
    sentences = text.split(". ")
    chunks, current = [], ""
    for s in sentences:
        if len(tokenizer.encode(current + s)) < max_tokens:
            current += s + ". "
        else:
            chunks.append(current.strip())
            current = s + ". "
    if current:
        chunks.append(current.strip())
    return chunks

# Анализ всей новости по конкретной акции
def classify_stock_in_news(news_text, stock_name):
    chunks = split_text(news_text)
    sentiments = [classify_sentiment_for_stock(chunk, stock_name) for chunk in chunks]
    sentiments = [s for s in sentiments if s != "Не определено"]

    if sentiments:
        return Counter(sentiments).most_common(1)[0][0]
    return "Не определено"


In [ ]:
news = '''Акции ЛУКОЙЛа подешевели на открытии биржевых торгов 17 декабря на 6,8%, до ₽6318 за бумагу, свидетельствуют данные Мосбиржи на 10:00 мск.
В понедельник, 16 декабря, был последний день, когда можно было купить акции ЛУКОЙЛа под дивиденды  . Сегодня акции компании открылись снижением на фоне дивидендного гэпа  .
Компания выплатит акционерам промежуточные дивиденды в размере ₽514 на акцию. Такая выплата дает доходность 7,58% по цене закрытия торгов понедельника.
Размер гэпа обычно равен размеру дивидендов, скорректированному на налог, пишут эксперты «Альфа-Инвестиций». По их расчетам, падение котировок может составить 6,5–7%. В брокере считают, что бумаги ЛУКОЙЛа имеют неплохие шансы на быстрое закрытие гэпа. По оценке аналитиков, в среднем у ЛУКОЙЛа это занимает 25 рабочих дней.
«Провал будет серьезный, и иной акции на его ликвидацию могло бы потребоваться много времени, но в ЛУКОЙЛе все может быть куда быстрее», — также уверен эксперт «БКС Мир инвестиций» Михаил Зельцер. Он подсчитал, что на фоне повышенных дивидендных ожиданий и с учетом статистики перекрытия дивгэпов курс вернется на исходный уровень 16 декабря еще до февраля 2025 года.
Вес акций ЛУКОЙЛа в индексе Мосбиржи составляет 18,19%, дивидендный гэп по ним в том числе стал причиной снижения бенчмарка — 17 декабря он опустился ниже 2400 пунктов. Кроме того, как отметил руководитель направления инвестиционного консультирования ИК «Алор Брокер» Алексей Антонов, свой вклад в снижение рынка акций внес и дивидендный гэп «Северстали». Вес бумаг в индексе Мосбиржи составляет 1,75%, на открытии торгов они открылись гэпом вниз на 4,65%.
'''
print("ЛУКОЙЛ:", classify_stock_in_news(news, "ЛУКОЙЛ"))
print("Татнефть:", classify_stock_in_news(news, "Татнефть"))

## ruBERT classificator

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from collections import Counter

# Загружаем rubert-sentiment
model_name = "blanchefort/rubert-base-cased-sentiment"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name).eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Классификация тональности текста
def classify_sentiment(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=2048).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.nn.functional.softmax(logits, dim=1)
    labels = ["Негативный", "Нейтральный", "Позитивный"]
    return labels[torch.argmax(probs)]

# Разбивка текста на куски (если новость длинная)
def split_text(text, max_chars=1000):
    sentences = text.split(". ")
    chunks, current = [], ""
    for s in sentences:
        if len(current + s) < max_chars:
            current += s + ". "
        else:
            chunks.append(current.strip())
            current = s + ". "
    if current:
        chunks.append(current.strip())
    return chunks

# Анализ новости по конкретной компании
def classify_company_in_news(news_text, company_name):
    chunks = split_text(news_text)
    sentiments = []

    for chunk in chunks:
        if company_name.lower() in chunk.lower():
            sentiments.append(classify_sentiment(chunk))

    if sentiments:
        return Counter(sentiments).most_common(1)[0][0]
    else:
        return "Н/Д"


In [ ]:
news = 'ВТБ сообщил о росте выручки. Роснефть объявила о сокращении расходов. VTB announced higher revenue; Rosneft reported lower costs.'

company_1 = "ЛУКОЙЛ"
company_2 = "Татнефть"

print(f"{company_1}:", classify_company_in_news(news, company_1))
print(f"{company_2}:", classify_company_in_news(news, company_2))


In [ ]:
import pandas as pd
All_news_LLM_mapped_merged = pd.read_csv(r'../../Data/Processed_data/All_news_firms_LLM_mapping/All_news_LLM_mapped_merged.csv',encoding='utf-16', sep='^')

In [ ]:
import ast

# Создаем новый список строк, где каждая фирма из строки 'firm_list'
# будет представлена в отдельной строке в колонке 'firm_name'
expanded_rows = []
for _, row in All_news_LLM_mapped_merged.iterrows():
    firms = row['firm_list']
    # Проверяем, является ли значение строкой
    if isinstance(firms, str):
        try:
            # Преобразуем строку в список с помощью ast.literal_eval
            firms_list = ast.literal_eval(firms)
            for company in firms_list:
                expanded_rows.append({**row, 'firm_name': company})
        except (ValueError, SyntaxError):
            # Если преобразование не удалось, добавляем строку как есть
            expanded_rows.append({**row, 'firm_name': firms})
    else:
        expanded_rows.append({**row, 'firm_name': firms})

# Создаем новый DataFrame из развернутых строк
All_news_LLM_mapped_merged = pd.DataFrame(expanded_rows)


In [ ]:
All_news_LLM_mapped_merged['LLM_sentiment'] = All_news_LLM_mapped_merged.apply(
    lambda row: classify_company_in_news(row['news_content'], row['firm_name']), axis=1
)

## Какие компании упоминаются в новости

In [ ]:
import os
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Загрузка модели
model_name = "sberbank-ai/rugpt3small_based_on_gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def is_company_mentioned(news_text, company):
    """Проверка: посвящена ли новость конкретной компании"""
    prompt = f"""
Прочитай новость и ответь "да" или "нет": новость посвящена компании "{company}"?

Новость:
\"\"\"{news_text}\"\"\"

Ответ (да или нет):"""
    
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1800).to(device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=5, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    
    response = tokenizer.decode(output[0], skip_special_tokens=True).lower()
    return "да" in response

def extract_relevant_companies(news_text, company_list):
    """Возвращает список компаний, которым посвящена новость"""
    relevant = []
    for company in company_list:
        if company.lower() in news_text.lower():  # предварительная фильтрация
            if is_company_mentioned(news_text, company):
                relevant.append(company)
    return relevant


In [ ]:
import pandas as pd
news = """
Акции Полюса подорожали на фоне отчета. Аналитики сравнивают динамику с Сургутнефтегазом и Газпромом.
"""
companies = pd.read_csv("names.csv", encoding= 'utf-16')['name'].tolist()
companies.remove('ГАЗ')

print("Компаниям посвящена новость:", extract_relevant_companies(news, companies))


## Новый меппинг имен компаний 

In [ ]:
import pandas as pd
All_news = pd.read_csv(r'../../Data/All_news_proccesed_merged.csv',
                       encoding='utf-16', sep='^').iloc[:,:5]

In [ ]:
companies = pd.read_csv("names.csv", encoding= 'utf-16')['name'].tolist()
companies.remove('ГАЗ')

In [ ]:
import os
from tqdm import tqdm

chunk_size = 100
start_chunk = 0  # начинаем с 30-го чанка

output_dir = r'../../Data/Processed_data/All_news_firms_LLM_mapping'
os.makedirs(output_dir, exist_ok=True)

if 'firm_list' not in All_news.columns:
    All_news['firm_list'] = pd.NA

start_idx = start_chunk * chunk_size

for i, start in enumerate(
    tqdm(range(start_idx, len(All_news), chunk_size), desc="Processing chunks"),
    start=start_chunk
):
    end = start + chunk_size
    chunk = All_news.iloc[start:end].copy()
    
    for idx in chunk.index:
        if not isinstance(chunk.at[idx, 'firm_list'], list):
            chunk.at[idx, 'firm_list'] = extract_relevant_companies(
                chunk.at[idx, 'news_content'], companies
            )
    
    output_file = os.path.join(output_dir, f"{i}.csv")
    chunk.to_csv(output_file, index=False, encoding='utf-16', sep='^')

In [ ]:
# from tqdm import tqdm

# if 'firm_list' not in All_news.columns:
#     All_news['firm_list'] = pd.NA

# for idx in tqdm(All_news.index, desc="Processing news"):
#     value = All_news.at[idx, 'firm_list']

#     if not isinstance(value, list):
#         All_news.at[idx, 'firm_list'] = extract_relevant_companies(All_news.at[idx, 'news_content'], companies)

In [ ]:
# All_news.to_csv(r'../../Data/Processed_data/All_news_firms_LLM_mapping/first_3067_rows.csv', encoding='utf-16', sep= '^')

In [ ]:
import os
import pandas as pd

csv_files = [os.path.join(output_dir, f) for f in os.listdir(output_dir) if f.lower().endswith('.csv')]

dfs = []
for file in csv_files:
    df = pd.read_csv(file,encoding='utf-16', sep= '^')  # Adjust encoding or separator if needed
    dfs.append(df)
    print(file)

In [ ]:
combined_df = pd.concat(dfs, ignore_index=True)

In [ ]:
# combined_df.to_csv(r'../../Data/Processed_data/All_news_firms_LLM_mapping/All_news_LLM_mapped_merged.csv', index=False, encoding='utf-16', sep= '^')